In [1]:
import pandas as pd
import geopandas as gpd

In [2]:
data_url = "https://data.wprdc.org/datastore/dump/2c13021f-74a9-4289-a1e5-fe0472c89881"
crashes = pd.read_csv(data_url, low_memory=False)

crashes["DEC_LAT"] = pd.to_numeric(crashes["DEC_LAT"], errors="coerce")
crashes["DEC_LONG"] = pd.to_numeric(crashes["DEC_LONG"], errors="coerce")

crashes = crashes.dropna(subset=["DEC_LAT", "DEC_LONG"])

crashes = crashes[crashes["DEC_LAT"].between(40.360, 40.520) & 
    crashes["DEC_LONG"].between(-80.100, -79.850)]
print("Crashes inside Pittsburgh area:", len(crashes))

neighborhood_url = ("https://services1.arcgis.com/YZCmUqbcsUpOKfj7/arcgis/rest/services/"
    "PGHWebNeighborhoods/FeatureServer/0/query"
    "?where=1%3D1&outFields=hood&returnGeometry=true&outSR=4326&f=geojson")

neighborhoods = gpd.read_file(neighborhood_url)

crashes_geo = gpd.GeoDataFrame(crashes,
    geometry=gpd.points_from_xy(crashes["DEC_LONG"],crashes["DEC_LAT"]),
    crs="EPSG:4326")


neighborhoods = neighborhoods.to_crs("EPSG:4326")

crash_to_neighborhood = gpd.sjoin(crashes_geo,
    neighborhoods,
    how="left",
    predicate="within")

neighborhood_counts = crash_to_neighborhood.groupby("hood").size()

neighborhood_counts = neighborhood_counts.sort_values()

print("Crashes in Each Neighborhood")
for neighborhood, count in neighborhood_counts.items():
    print(neighborhood, " -> ", count)

Crashes inside Pittsburgh area: 122881
Crashes in Each Neighborhood
Arlington Heights  ->  9
Mt. Oliver  ->  44
East Carnegie  ->  58
Chartiers City  ->  66
New Homestead  ->  78
Glen Hazel  ->  90
Oakwood  ->  100
Fairywood  ->  131
Regent Square  ->  133
Spring Garden  ->  142
St. Clair  ->  163
Windgap  ->  165
California-Kirkbride  ->  168
Bedford Dwellings  ->  175
Upper Hill  ->  219
Arlington  ->  230
Fineview  ->  231
Terrace Village  ->  249
Ridgemont  ->  273
Manchester  ->  286
Allegheny Center  ->  287
Allegheny West  ->  303
Esplen  ->  304
Friendship  ->  354
Upper Lawrenceville  ->  356
Westwood  ->  356
Northview Heights  ->  393
Central Northside  ->  397
Middle Hill  ->  404
Morningside  ->  446
Crafton Heights  ->  448
Spring Hill-City View  ->  479
West Oakland  ->  499
Stanton Heights  ->  505
Allentown  ->  508
Lincoln Place  ->  527
Beltzhoover  ->  545
Bon Air  ->  566
Sheraden  ->  570
Knoxville  ->  589
Homewood North  ->  590
Garfield  ->  591
Polish Hill  ->